# 09 · Practical D — Generating Tenor Dates

**Read first:** Chapter 10 (*Vanilla FX Derivatives Miscellaneous Topics*), then Practical D.

---

## What you'll be able to do after this

- Compute the expiry and delivery date for any market tenor, and say why weeks and months follow different rules.
- Explain what the four dates are and which one each cash flow attaches to.
- Name the conventions this implementation skips, and what they would change.

## The intuition, before the maths

This is the least glamorous practical and everything in Part II depends on it.

Here's the problem. A trader says "one month ATM". That is not a contract — it's a *label*. To price anything you need an actual date, and the date has to be the one the whole market agrees on, or you and your counterparty have priced different things.

### The bit that surprises people

Tenors are fixed but dates are not. **Today's 1-month contract has a different expiry date from yesterday's.** The liquid instruments roll forward every single day. That's what makes an OTC market OTC — nothing is standardised, and the price you quote today for "1 month" is for a contract that did not exist yesterday and will not exist tomorrow.

### Why not just add 30 days?

Because an FX option ends in a cash exchange, and cash settlement is what the convention is built around.

Think about what actually happens. The option expires. Two business days later the currencies change hands — same lag as a spot trade, for the same reason: banks need time to move money. So the **delivery date is what matters commercially**, and the expiry date is derived from it by working backwards.

That's the whole rule:

- **Weeks** are short enough that nobody bothers — add them to today and move on.
- **Months and years** are defined by their delivery date. Go out to the spot date, add the months to get the delivery date, then come back two business days to find the expiry.

One is a shortcut, the other is the real convention. Practical D implements both.

## The maths, derived not asserted

Almost none. It's a set of rules, and the value is in knowing *why* each exists.

### The four dates

```
horizon ──2bd──▶ spot date          (premium and spot hedges settle)
   │
   │  the contract's life
   ▼
expiry ───2bd──▶ delivery date      (final funds move, forward hedges settle)
```

**The symmetry is the point.** Delivery is derived from expiry exactly as spot is derived from horizon. Same lag, same rule. Once you see that, the month-tenor logic stops looking arbitrary: it's just "pick the delivery date first, then work back to the expiry that produces it".

All four can only be weekdays. The FX market is shut at the weekend.

### Business day arithmetic

The book special-cases the weekend: Saturday +2, Friday +3, otherwise +1. That's the same as "step forward until you land on a business day", which also handles holidays once you have a calendar. This module does the latter.

### The tenor rules

| Tenor | Rule |
|---|---|
| `ON` | Next business day after the horizon |
| `nD`, `nW` | Horizon **+ n or 7n calendar days** |
| `nM` | Spot date + n months → roll to a business day → back 2bd |
| `nY` | Spot date + n years → roll to a business day → back 2bd |

### Discounting, while we're here

Chapter 10 also sets out the discount factor conventions. The one that matters for this repo:

$$df = e^{-r \cdot T}$$

the continuously compounded form, which is what the Black-Scholes framework assumes. The book also lists zero rates ($1/(1+r_0T)$), annual compounding ($1/(1+r_A)^T$) and the general $m$-times-a-year case — and then says real curve building is bootstrapped from many instruments and is not a day-to-day concern for an FX derivatives trader. We take the same view.

## The code

In [1]:
from datetime import date, timedelta
import pandas as pd

from fxds.dates import (
    tenor_table, tenor_dates, expiry_from_tenor,
    spot_date_from_horizon, horizon_from_spot_date, delivery_date_from_expiry,
    next_business_day, previous_business_day, is_business_day,
    validate_day_week_expiry, InvalidTenorError,
)
from fxds.conventions import MARKET_TENORS

# The book's worked horizon: Wednesday 11 June 2014.
# (Excel stores this as serial 41801, which is why the book keeps mentioning
# forty-thousand-something.)
HORIZON = date(2014, 6, 11)
print(f"horizon    {HORIZON}  {HORIZON.strftime('%A')}")
print(f"spot date  {spot_date_from_horizon(HORIZON)}  "
      f"{spot_date_from_horizon(HORIZON).strftime('%A')}")

horizon    2014-06-11  Wednesday
spot date  2014-06-13  Friday


### Business day arithmetic — the weekend cases

In [2]:
week = [date(2014, 6, d) for d in range(9, 16)]
rows = [{"date": d, "day": d.strftime("%a"),
         "business day?": is_business_day(d),
         "next bd": next_business_day(d),
         "previous bd": previous_business_day(d)} for d in week]
print(pd.DataFrame(rows).to_string(index=False))

      date day  business day?    next bd previous bd
2014-06-09 Mon           True 2014-06-10  2014-06-06
2014-06-10 Tue           True 2014-06-11  2014-06-09
2014-06-11 Wed           True 2014-06-12  2014-06-10
2014-06-12 Thu           True 2014-06-13  2014-06-11
2014-06-13 Fri           True 2014-06-16  2014-06-12
2014-06-14 Sat          False 2014-06-16  2014-06-13
2014-06-15 Sun          False 2014-06-16  2014-06-13


Friday jumps three days forward, Monday jumps three back, Saturday and Sunday collapse onto the nearest weekday. Exactly the special cases the book's VBA enumerates.

### The tenor table — the deliverable of Practical D

In [3]:
table = tenor_table(HORIZON)
print(table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

tenor     expiry expiry_day   delivery delivery_day  days  years
   ON 2014-06-12        Thu 2014-06-16          Mon     1 0.0027
   1W 2014-06-18        Wed 2014-06-20          Fri     7 0.0192
   2W 2014-06-25        Wed 2014-06-27          Fri    14 0.0384
   1M 2014-07-10        Thu 2014-07-14          Mon    29 0.0795
   2M 2014-08-11        Mon 2014-08-13          Wed    61 0.1671
   3M 2014-09-11        Thu 2014-09-15          Mon    92 0.2521
   6M 2014-12-11        Thu 2014-12-15          Mon   183 0.5014
   9M 2015-03-11        Wed 2015-03-13          Fri   273 0.7479
   1Y 2015-06-11        Thu 2015-06-15          Mon   365 1.0000
   2Y 2016-06-09        Thu 2016-06-13          Mon   729 1.9973


Two things worth pausing on.

**The 1M expiry is 10 July, not 11 July.** Follow the chain: spot is Friday 13 June → plus one month is Sunday 13 July → roll forward to Monday 14 July → back two business days to Thursday 10 July. Every step is forced by the convention, and adding a month to the horizon would have given you the wrong date.

**1M and 4W are different contracts.** Not "roughly the same" — computed by different rules entirely.

In [4]:
for tenor in ("4W", "1M"):
    d = tenor_dates(HORIZON, tenor)
    print(f"{tenor}: expiry {d.expiry} ({d.expiry_weekday}), {d.days_to_expiry} days")

print("\nWhy: 4W is horizon + 28 days. 1M goes out to the delivery date and comes back.")
spot = spot_date_from_horizon(HORIZON)
from dateutil.relativedelta import relativedelta
delivery = spot + relativedelta(months=1)
print(f"\n  spot date        {spot}  {spot.strftime('%a')}")
print(f"  + 1 month        {delivery}  {delivery.strftime('%a')}   <- a Sunday")
rolled = delivery
while not is_business_day(rolled):
    rolled += timedelta(days=1)
print(f"  roll to business {rolled}  {rolled.strftime('%a')}")
print(f"  back 2 bd        {horizon_from_spot_date(rolled)}  "
      f"{horizon_from_spot_date(rolled).strftime('%a')}   <- the 1M expiry")

4W: expiry 2014-07-09 (Wed), 28 days
1M: expiry 2014-07-10 (Thu), 29 days

Why: 4W is horizon + 28 days. 1M goes out to the delivery date and comes back.

  spot date        2014-06-13  Fri
  + 1 month        2014-07-13  Sun   <- a Sunday
  roll to business 2014-07-14  Mon
  back 2 bd        2014-07-10  Thu   <- the 1M expiry


### Invalid tenors

The book pops a message box and returns `-1`. A sentinel that is also a plausible-looking date serial is a bug waiting to happen — Practical E's own code then has to test for it. This raises instead.

In [5]:
for bad in ["banana", "1X", "M1", "0M", ""]:
    try:
        expiry_from_tenor(HORIZON, bad)
        print(f"  {bad!r:10s} -> no error (unexpected)")
    except InvalidTenorError as exc:
        print(f"  {bad!r:10s} -> {type(exc).__name__}: {str(exc)[:58]}...")

  'banana'   -> InvalidTenorError: Cannot parse tenor 'banana'. Expected 'ON' or a count and ...
  '1X'       -> InvalidTenorError: Cannot parse tenor '1X'. Expected 'ON' or a count and unit...
  'M1'       -> InvalidTenorError: Cannot parse tenor 'M1'. Expected 'ON' or a count and unit...
  '0M'       -> InvalidTenorError: Tenor count must be positive, got '0M'...
  ''         -> InvalidTenorError: Cannot parse tenor ''. Expected 'ON' or a count and unit s...


## Experiments

### Experiment 1 — How much do the dates move as the horizon rolls?

**Predict:** you quote a 1M price on Monday and again on Tuesday. How far apart are the two expiry dates? Always one day?

In [6]:
rows = []
for offset in range(10):
    h = HORIZON + timedelta(days=offset)
    if not is_business_day(h):
        continue
    e = expiry_from_tenor(h, "1M")
    rows.append({"horizon": h, "h_day": h.strftime("%a"),
                 "1M expiry": e, "e_day": e.strftime("%a"),
                 "days to expiry": (e - h).days})
frame = pd.DataFrame(rows)
frame["expiry moved"] = frame["1M expiry"].diff().dt.days if hasattr(
    frame["1M expiry"].diff(), "dt") else [None] + [
    (frame["1M expiry"].iloc[i] - frame["1M expiry"].iloc[i-1]).days
    for i in range(1, len(frame))]
print(frame.to_string(index=False))

   horizon h_day  1M expiry e_day  days to expiry  expiry moved
2014-06-11   Wed 2014-07-10   Thu              29           NaN
2014-06-12   Thu 2014-07-14   Mon              32           4.0
2014-06-13   Fri 2014-07-15   Tue              32           1.0
2014-06-16   Mon 2014-07-16   Wed              30           1.0
2014-06-17   Tue 2014-07-17   Thu              30           1.0
2014-06-18   Wed 2014-07-17   Thu              29           0.0
2014-06-19   Thu 2014-07-21   Mon              32           4.0
2014-06-20   Fri 2014-07-22   Tue              32           1.0


**Result:** mostly one day, but not always — the expiry jumps three days across a weekend, and the *days to expiry* wobbles between 29 and 32 rather than staying fixed.

That wobble matters. Time to expiry feeds straight into every Black-Scholes calculation, so the "same" 1M contract has a slightly different `T` every day. Combined with Chapter 11's point that the market prices in **whole days**, this is why short-dated implied volatility drifts down through a trading day and jumps back up at the start of the next one.

### Experiment 2 — The end-of-month special cases the practical skips

Chapter 10 sets out two conventions Practical D explicitly ignores. Both bite at month ends.

**Predict:** the spot date is 30 January and the tenor is 1M. February has no 30th. What should the delivery date be?

In [7]:
from dateutil.relativedelta import relativedelta

print("Chapter 10, special case 2 — month overflow:\n")
for spot_day in [date(2014, 1, 28), date(2014, 1, 29), date(2014, 1, 30), date(2014, 1, 31)]:
    natural = spot_day + relativedelta(months=1)
    print(f"  spot {spot_day} + 1M -> {natural}  ({natural.strftime('%a')})")

print("\nrelativedelta clamps 29/30/31 January all onto 28 February.")
print("That happens to agree with the convention (last business day of the target")
print("month) but it gets there by arithmetic accident, not by implementing the rule.")
print("\nChapter 10, special case 1 — end-end:")
print("  If the spot date is the LAST BUSINESS DAY of its month, the delivery date")
print("  is by convention the last business day of the target month.")
print("  So spot 30 April + 1M should give delivery 31 May, not 30 May.")
print("\nNeither rule is implemented here - both are marked TODO in fxds/dates.py")
print("and recorded in notes/deviations.md. They are real and they matter at")
print("month ends, which is exactly when volumes are highest.")

Chapter 10, special case 2 — month overflow:

  spot 2014-01-28 + 1M -> 2014-02-28  (Fri)
  spot 2014-01-29 + 1M -> 2014-02-28  (Fri)
  spot 2014-01-30 + 1M -> 2014-02-28  (Fri)
  spot 2014-01-31 + 1M -> 2014-02-28  (Fri)

relativedelta clamps 29/30/31 January all onto 28 February.
That happens to agree with the convention (last business day of the target
month) but it gets there by arithmetic accident, not by implementing the rule.

Chapter 10, special case 1 — end-end:
  If the spot date is the LAST BUSINESS DAY of its month, the delivery date
  is by convention the last business day of the target month.
  So spot 30 April + 1M should give delivery 31 May, not 30 May.

Neither rule is implemented here - both are marked TODO in fxds/dates.py
and recorded in notes/deviations.md. They are real and they matter at
month ends, which is exactly when volumes are highest.


**Result:** the arithmetic happens to land in the right place for overflow and in the *wrong* place for end-end. That distinction is worth internalising: code that accidentally agrees with a convention will diverge from it the moment the inputs change.

### Experiment 3 — What a holiday calendar would change

Practical D handles weekends only. The seam for a calendar is left open, so we can measure what it would do.

**Predict:** add US Independence Day (Friday 4 July 2014) as a holiday. Which tenors move?

In [8]:
holidays = {date(2014, 12, 25), date(2014, 12, 26), date(2015, 1, 1)}
cal = holidays.__contains__

# The horizon matters. From 11 June the Christmas holidays fall outside every
# tenor's spot and delivery window, so NOTHING moves - which is a real and slightly
# boring answer. Move the horizon to just before Christmas and they bite properly.
XMAS_HORIZON = date(2014, 12, 23)
print(f"horizon {XMAS_HORIZON} ({XMAS_HORIZON.strftime('%a')}), "
      f"holidays {sorted(holidays)}\n")

plain = tenor_table(XMAS_HORIZON)[["tenor", "expiry", "delivery"]]
with_hols = tenor_table(XMAS_HORIZON, calendar=cal)[["tenor", "expiry", "delivery"]]

merged = plain.merge(with_hols, on="tenor", suffixes=("_weekends", "_holidays"))
merged["expiry moved"] = merged["expiry_holidays"] != merged["expiry_weekends"]
merged["delivery moved"] = merged["delivery_holidays"] != merged["delivery_weekends"]
print(merged.to_string(index=False))
print(f"\n{merged['expiry moved'].sum()} expiries moved, "
      f"{merged['delivery moved'].sum()} deliveries moved")

# And for contrast, the original June horizon.
june_plain = tenor_table(HORIZON)[["tenor", "expiry", "delivery"]]
june_hols = tenor_table(HORIZON, calendar=cal)[["tenor", "expiry", "delivery"]]
june = june_plain.merge(june_hols, on="tenor", suffixes=("_a", "_b"))
moved = ((june["expiry_a"] != june["expiry_b"]) | (june["delivery_a"] != june["delivery_b"])).sum()
print(f"From the June horizon, by contrast: {moved} dates moved.")

horizon 2014-12-23 (Tue), holidays [datetime.date(2014, 12, 25), datetime.date(2014, 12, 26), datetime.date(2015, 1, 1)]

tenor expiry_weekends delivery_weekends expiry_holidays delivery_holidays  expiry moved  delivery moved
   ON      2014-12-24        2014-12-26      2014-12-24        2014-12-30         False            True
   1W      2014-12-30        2015-01-01      2014-12-30        2015-01-02         False            True
   2W      2015-01-06        2015-01-08      2015-01-06        2015-01-08         False           False
   1M      2015-01-22        2015-01-26      2015-01-27        2015-01-29          True            True
   2M      2015-02-23        2015-02-25      2015-02-26        2015-03-02          True            True
   3M      2015-03-23        2015-03-25      2015-03-26        2015-03-30          True            True
   6M      2015-06-23        2015-06-25      2015-06-25        2015-06-29          True            True
   9M      2015-09-23        2015-09-25      2

**Result:** it depends entirely on where the horizon sits, which is itself the lesson.

From the June horizon the Christmas holidays fall outside every tenor's window and **nothing moves at all**. Move the horizon to 23 December and 7 expiries and 9 deliveries shift.

Note that **expiries do move here**, which contradicts the tidy story I was about to tell. Chapter 10 says an expiry *may* fall on a holiday — it only has to be a weekday, and only 1 January is excluded outright. But the month and year expiries are derived by stepping back two **business** days from the delivery date, and a holiday is not a business day. So a holiday near the delivery date drags the expiry with it, even though the expiry itself was allowed to sit on a holiday.

The clean mental model still holds underneath: **expiry is when a decision gets made, delivery is when cash moves, and only cash movement is blocked by a bank holiday.** But because month expiries are *defined backwards from* delivery, blocking the cash date moves the decision date too.

Chapter 10 adds a further wrinkle we do not implement: for most currencies nothing settles on a **US** holiday even when USD is not in the pair.

In [9]:
saturday = date(2014, 6, 14)
expiry = expiry_from_tenor(saturday, "1W")
print(f"horizon {saturday} ({saturday.strftime('%a')}) + 1W = "
      f"{expiry} ({expiry.strftime('%a')})")
print("expiry_from_tenor returned it without complaint - following the practical's code.\n")

try:
    validate_day_week_expiry(expiry)
except InvalidTenorError as exc:
    print(f"validate_day_week_expiry says: {exc}")

horizon 2014-06-14 (Sat) + 1W = 2014-06-21 (Sat)
expiry_from_tenor returned it without complaint - following the practical's code.

validate_day_week_expiry says: Expiry 2014-06-21 falls on a Saturday. Chapter 10 makes this tenor invalid; Practical D's code does not check it.


**Result:** the module follows the practical's code and exposes the chapter's stricter rule separately, so you can apply it when you want it.

This is a deliberate choice and it is recorded in `notes/deviations.md`. The book contradicts itself here — the prose in Chapter 10 and the VBA in Practical D disagree — and quietly picking one without saying so would leave you unable to reconcile the code with the text.

## Common misconceptions

**"Tenor, expiry and delivery are three words for the same thing."**
Tenor is a *label* (`"1M"`). Expiry is the **date** the option is exercised or abandoned. Delivery is the **date the cash moves**, two business days after expiry. Different things, different dates, different purposes.

**"1M means 30 days."**
It means whatever the delivery-date convention produces — 29 days in this example, and it changes daily as the horizon rolls.

**"1M ≈ 4W."**
Different rules entirely. One goes via the spot date and back; the other is added to the horizon. They land on different dates and they are different contracts.

**"The expiry date can't be a holiday."**
It can. Chapter 10: an expiry can be any weekday even if it is a holiday in one or both currencies — except 1 January. It is the *delivery* date that must avoid holidays, because that is when money clears.

**"The spot date is always two days after the horizon."**
Two *business* days, and some pairs are T+1 (USD/CAD, USD/TRY). Chapter 10 adds that for T+2 there must be one clear working day for USD and two for everything else, and that nothing settles on a US holiday even in non-USD pairs. None of that is implemented here.

**"Once you've computed the expiry, it's fixed."**
Chapter 10's closing note: expiry dates can change *within* a trading day. USD/JPY traded for Tokyo cut in Asian hours may use a different expiry from the same tenor traded for NY cut once London is in.

## Check yourself

1. Horizon is Wednesday. Why is the 1M expiry a Thursday rather than a Wednesday?
2. Which is longer, `4W` or `1M`, and can you be sure without computing it?
3. An option expires on Friday 12 December. When does the cash move?
4. Why can a week tenor be added to the horizon but a month tenor cannot?
5. Your 1Y expiry lands on a Saturday. What went wrong?

In [10]:
#@title Answers — run this cell to reveal
from IPython.display import Markdown
Markdown(r'''
**1.** The expiry is derived from the delivery date, not the horizon. Spot is Friday 13 June; plus a month is Sunday 13 July; roll forward to Monday 14 July; back two business days lands on Thursday 10 July. The weekday of the horizon has no direct influence.

**2.** **You cannot be sure** — it depends where the weekends fall. Here 1M is 29 days and 4W is 28, but a month tenor rolling across an awkward weekend can come out shorter than four weeks. They are computed by different rules, so any relationship between them is a coincidence of the calendar.

**3.** **Tuesday 16 December.** Delivery is two business days after expiry, and the weekend is skipped — the same rule that takes a horizon to its spot date.

**4.** Because month contracts are **defined by their delivery date**. The market convention fixes when the cash moves, and the expiry is whatever date produces that delivery. Weeks are short enough that the market does not bother, and adds them to the horizon directly.

**5.** Nothing — check your inputs. A year tenor goes via the delivery date and comes back two *business* days, so it always lands on a weekday. If you have a Saturday you have used a week or day tenor, where the practical's code does no check at all. Chapter 10 says such a tenor is invalid; `validate_day_week_expiry` will tell you so.
''')


**1.** The expiry is derived from the delivery date, not the horizon. Spot is Friday 13 June; plus a month is Sunday 13 July; roll forward to Monday 14 July; back two business days lands on Thursday 10 July. The weekday of the horizon has no direct influence.

**2.** **You cannot be sure** — it depends where the weekends fall. Here 1M is 29 days and 4W is 28, but a month tenor rolling across an awkward weekend can come out shorter than four weeks. They are computed by different rules, so any relationship between them is a coincidence of the calendar.

**3.** **Tuesday 16 December.** Delivery is two business days after expiry, and the weekend is skipped — the same rule that takes a horizon to its spot date.

**4.** Because month contracts are **defined by their delivery date**. The market convention fixes when the cash moves, and the expiry is whatever date produces that delivery. Weeks are short enough that the market does not bother, and adds them to the horizon directly.

**5.** Nothing — check your inputs. A year tenor goes via the delivery date and comes back two *business* days, so it always lands on a weekday. If you have a Saturday you have used a week or day tenor, where the practical's code does no check at all. Chapter 10 says such a tenor is invalid; `validate_day_week_expiry` will tell you so.


## Where next

**Notebook 10 — Practical E** builds the ATM curve on top of these dates. That is where the saw-tooth appears, and it is the moment most of Part II clicks into place.

Everything from here needs a date. If these functions are wrong, every volatility downstream is wrong in a way that is very hard to spot — which is why the least glamorous practical comes before the interesting ones.